# Trigger Classifier — Stage 2

**Approach:** 3-class Random Forest trained on pre-onset window features.

**Features:** Selected from exploratory analysis — 7 features with demonstrated separation between trigger classes.

**Evaluation:** Stratified 5-fold CV, per-class F1 + macro F1 + confusion matrix.

**Limitation acknowledged:** StartHesitation (n=86) and Walking (n=79) have small positive sample counts. Results for these classes should be interpreted as preliminary.

**No accuracy reported** — misleading given 83% Turn dominance.

## 0. Imports & Config

In [ ]:
import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    f1_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import LabelEncoder
import joblib

# ── Config (must match analysis notebook exactly) ─────────────────
DATA_DIR     = "tdcsfog_csvs"
FS           = 128
WINDOW_SIZE  = 192
SAFETY_GAP   = 64           # 0.5 s before episode start
FOG_CLEAN_THRESH = 0.2

AXES         = ["AccV", "AccML", "AccAP"]
TRIGGER_COLS = ["StartHesitation", "Turn", "Walking"]
TRIGGER_TYPES = ["StartHesitation", "Turn", "Walking"]
FREQS        = np.fft.rfftfreq(WINDOW_SIZE, d=1/FS)

# ── Selected features from exploratory analysis ───────────────────
SELECTED_FEATURES = [
    "var_AccAP",           # strongest SH separator (4.3x ratio)
    "dom_freq_AccV",       # SH ~1.8Hz vs Turn/Walk ~3.8Hz
    "dom_freq_AccAP",      # supporting frequency evidence
    "corr_AccV_AccAP",     # SH positive, Turn negative
    "mean_abs_jerk_AccV",  # SH pre-deceleration
    "mean_abs_jerk_AccML", # supporting jerk evidence
    "energy_AccAP",        # SH and Walk higher than Turn
]

COLORS = {
    "StartHesitation": "#e63946",
    "Turn":            "#457b9d",
    "Walking":         "#2a9d8f"
}

print("Config ready.")
print(f"Selected features: {SELECTED_FEATURES}")

## 1. Feature Extraction

Same pre-onset extraction logic as the analysis notebook. Memory-safe: features computed on-the-fly, raw windows discarded immediately.

In [ ]:
def find_episodes(df):
    fog = (
        (df["StartHesitation"] == 1) |
        (df["Turn"]            == 1) |
        (df["Walking"]         == 1)
    ).astype(int).values

    sh = df["StartHesitation"].values
    tu = df["Turn"].values
    wa = df["Walking"].values

    episodes = []
    in_fog   = False
    ep_start = 0

    for i in range(len(fog)):
        if fog[i] == 1 and not in_fog:
            ep_start = i
            in_fog   = True
        elif fog[i] == 0 and in_fog:
            ep_end = i
            sh_c = sh[ep_start:ep_end].sum()
            tu_c = tu[ep_start:ep_end].sum()
            wa_c = wa[ep_start:ep_end].sum()
            trig = TRIGGER_TYPES[np.argmax([sh_c, tu_c, wa_c])]
            episodes.append((ep_start, ep_end, trig))
            in_fog = False

    if in_fog:
        ep_end = len(fog)
        sh_c = sh[ep_start:ep_end].sum()
        tu_c = tu[ep_start:ep_end].sum()
        wa_c = wa[ep_start:ep_end].sum()
        trig = TRIGGER_TYPES[np.argmax([sh_c, tu_c, wa_c])]
        episodes.append((ep_start, ep_end, trig))

    return episodes


def compute_features(win, trigger):
    rec = {"trigger": trigger}

    for ax_idx, ax_name in enumerate(AXES):
        sig  = win[:, ax_idx]
        jerk = np.diff(sig)
        sig_detrended = sig - sig.mean()
        fft_mag = np.abs(np.fft.rfft(sig_detrended))

        rec[f"energy_{ax_name}"]            = float(np.mean(sig ** 2))
        rec[f"var_{ax_name}"]               = float(np.var(sig))
        rec[f"mean_abs_jerk_{ax_name}"]     = float(np.mean(np.abs(jerk)))
        rec[f"dom_freq_{ax_name}"]          = float(FREQS[np.argmax(fft_mag)])

    rec["corr_AccV_AccML"]  = float(np.corrcoef(win[:,0], win[:,1])[0,1])
    rec["corr_AccV_AccAP"]  = float(np.corrcoef(win[:,0], win[:,2])[0,1])
    rec["corr_AccML_AccAP"] = float(np.corrcoef(win[:,1], win[:,2])[0,1])

    return rec


print("Functions defined.")

In [ ]:
csv_files = [
    os.path.join(DATA_DIR, f)
    for f in os.listdir(DATA_DIR)
    if f.endswith(".csv")
]

required = set(AXES + TRIGGER_COLS)
records  = []

for path in tqdm(csv_files, desc="Extracting features"):
    df = pd.read_csv(path)
    if not required.issubset(df.columns):
        continue

    signals = df[AXES].values.astype(np.float32)
    fog_arr = (
        (df["StartHesitation"].values == 1) |
        (df["Turn"].values            == 1) |
        (df["Walking"].values         == 1)
    ).astype(np.float32)

    for (ep_start, ep_end, trig) in find_episodes(df):
        pre_end   = ep_start - SAFETY_GAP
        pre_start = pre_end  - WINDOW_SIZE
        if pre_start < 0:
            continue
        if fog_arr[pre_start:pre_end].mean() > FOG_CLEAN_THRESH:
            continue

        win = signals[pre_start:pre_end]
        rec = compute_features(win, trig)
        records.append(rec)
        del win

    del signals, fog_arr
    gc.collect()

feat_df = pd.DataFrame(records)

print(f"\nTotal windows  : {len(feat_df)}")
print("\nClass distribution:")
print(feat_df["trigger"].value_counts())

## 2. Prepare Feature Matrix

In [ ]:
X = feat_df[SELECTED_FEATURES].values
y = feat_df["trigger"].values

# Encode labels to integers (required by sklearn metrics)
le = LabelEncoder()
le.fit(TRIGGER_TYPES)   # fix order: StartHesitation=0, Turn=1, Walking=2
y_enc = le.transform(y)

print(f"X shape : {X.shape}")
print(f"Classes : {le.classes_}")
print(f"Encoded : {np.unique(y_enc, return_counts=True)}")

## 3. Stratified 5-Fold Cross-Validation

Stratified folds ensure each fold has the same class ratio as the full dataset. This is critical with small minority classes — a random split could put all Walking samples in one fold.

In [ ]:
clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",   # inversely proportional to class frequency
    random_state=42,
    n_jobs=-1
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Collect per-fold predictions for final aggregated metrics
all_true  = []
all_pred  = []
fold_f1s  = []   # macro F1 per fold

print("Running stratified 5-fold CV...\n")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_enc), 1):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y_enc[train_idx], y_enc[val_idx]

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_val)

    macro_f1 = f1_score(y_val, y_pred, average="macro", zero_division=0)
    per_class_f1 = f1_score(y_val, y_pred, average=None,
                             labels=[0,1,2], zero_division=0)

    fold_f1s.append(macro_f1)
    all_true.extend(y_val)
    all_pred.extend(y_pred)

    print(f"Fold {fold} | Macro F1: {macro_f1:.3f} | "
          f"SH: {per_class_f1[0]:.3f}  "
          f"Turn: {per_class_f1[1]:.3f}  "
          f"Walk: {per_class_f1[2]:.3f}")

print(f"\nMean Macro F1 : {np.mean(fold_f1s):.3f}  "
      f"(±{np.std(fold_f1s):.3f})")

## 4. Aggregated Classification Report

Combining predictions across all folds gives a more stable estimate than any single fold.

In [ ]:
all_true = np.array(all_true)
all_pred = np.array(all_pred)

print("=" * 55)
print("AGGREGATED CLASSIFICATION REPORT (all folds)")
print("=" * 55)
print(classification_report(
    all_true, all_pred,
    target_names=le.classes_,
    zero_division=0
))
print("NOTE: Accuracy not reported — misleading with 83% Turn dominance.")
print("Primary metric is Macro F1.")

## 5. Confusion Matrix

In [ ]:
cm = confusion_matrix(all_true, all_pred, labels=[0, 1, 2])

# Raw counts
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=le.classes_
)
disp.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Confusion Matrix — Raw Counts", fontsize=11)
axes[0].tick_params(axis='x', rotation=15)

# Normalised (row = true class)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
disp_norm = ConfusionMatrixDisplay(
    confusion_matrix=cm_norm,
    display_labels=le.classes_
)
disp_norm.plot(ax=axes[1], colorbar=False, cmap="Blues")
axes[1].set_title("Confusion Matrix — Normalised (recall per class)", fontsize=11)
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle("Trigger Classifier — Aggregated CV Results",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("trigger_classifier_cm.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: trigger_classifier_cm.png")

## 6. Feature Importance

Retrain on full dataset to get stable feature importance estimates.

In [ ]:
clf_full = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
clf_full.fit(X, y_enc)

importances = clf_full.feature_importances_
feat_names  = SELECTED_FEATURES

# Sort by importance
order = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(
    range(len(feat_names)),
    importances[order],
    color="#457b9d", alpha=0.85
)
ax.set_xticks(range(len(feat_names)))
ax.set_xticklabels([feat_names[i] for i in order],
                    rotation=30, ha='right', fontsize=9)
ax.set_ylabel("Mean Decrease in Impurity")
ax.set_title("Feature Importance — Trigger Classifier (full dataset)",
             fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig("trigger_feature_importance.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: trigger_feature_importance.png")

print("\nFeature importances:")
for i in order:
    print(f"  {feat_names[i]:<25} {importances[i]:.4f}")

## 7. Per-Class F1 Stability Plot

Shows how stable each class F1 is across folds. High variance = unreliable estimate (expected for minority classes).

In [ ]:
# Re-run CV collecting per-class F1 per fold
fold_per_class = {t: [] for t in le.classes_}
fold_macro     = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_enc), 1):
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y_enc[train_idx], y_enc[val_idx]

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_val)

    pf = f1_score(y_val, y_pred, average=None,
                  labels=[0,1,2], zero_division=0)
    for i, t in enumerate(le.classes_):
        fold_per_class[t].append(pf[i])
    fold_macro.append(f1_score(y_val, y_pred, average="macro", zero_division=0))

fig, ax = plt.subplots(figsize=(8, 4))
x_pos = np.arange(5)

for i, (trig, color) in enumerate(COLORS.items()):
    vals = fold_per_class[trig]
    ax.plot(x_pos, vals, marker='o', color=color,
            label=f"{trig} (mean={np.mean(vals):.2f})", linewidth=2)

ax.plot(x_pos, fold_macro, marker='s', color='black',
        linestyle='--', label=f"Macro F1 (mean={np.mean(fold_macro):.2f})",
        linewidth=2)

ax.set_xticks(x_pos)
ax.set_xticklabels([f"Fold {i+1}" for i in range(5)])
ax.set_ylabel("F1 Score")
ax.set_ylim(0, 1.05)
ax.set_title("Per-Class F1 Stability Across CV Folds",
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("trigger_f1_stability.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved: trigger_f1_stability.png")

## 8. Save Final Model & Summary

In [ ]:
# Save model trained on full dataset
joblib.dump(clf_full, "trigger_classifier.pkl")
joblib.dump(le,       "trigger_label_encoder.pkl")
print("Saved: trigger_classifier.pkl")
print("Saved: trigger_label_encoder.pkl")

# Save CV summary
summary = {
    "n_samples":         len(X),
    "n_features":        len(SELECTED_FEATURES),
    "features":          SELECTED_FEATURES,
    "class_distribution": feat_df["trigger"].value_counts().to_dict(),
    "cv_macro_f1_mean":  round(float(np.mean(fold_macro)), 4),
    "cv_macro_f1_std":   round(float(np.std(fold_macro)),  4),
    "cv_f1_per_class":   {t: round(float(np.mean(v)), 4)
                          for t, v in fold_per_class.items()},
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv("trigger_classifier_summary.csv", index=False)

print("\n" + "="*55)
print("FINAL SUMMARY")
print("="*55)
print(f"Samples          : {summary['n_samples']}")
print(f"Features         : {summary['n_features']}")
print(f"Class dist       : {summary['class_distribution']}")
print(f"CV Macro F1      : {summary['cv_macro_f1_mean']} "
      f"(±{summary['cv_macro_f1_std']})")
print(f"Per-class F1     :")
for t, v in summary['cv_f1_per_class'].items():
    n = feat_df[feat_df['trigger']==t].shape[0]
    flag = " ← preliminary (small n)" if n < 100 else ""
    print(f"  {t:<20} {v:.4f}  (n={n}){flag}")

## 9. Research Limitations Note

To be included in any write-up:

**Class imbalance:** The tdcsfog dataset reflects the natural clinical distribution of FOG trigger types under a lab protocol — Turn episodes are the most reliably provoked (83% of episodes). StartHesitation (n=86) and Walking (n=79) have limited pre-onset windows available.

**Interpretation:** Turn classification results are robust and generalisable. StartHesitation and Walking F1 scores should be interpreted as preliminary estimates with high variance across folds (visible in the stability plot).

**Future work:** The defog dataset (home recordings) may provide more balanced trigger distributions. Combining tdcsfog and defog pre-onset windows could improve minority class performance.